# BISINDO YOLO26 Classification — vocabulary expansion research

**Tahap 0: rencana prospektif, bukan training notebook final.** Tanggal 2026-11-03, Asia/Bangkok.
Notebook baru ini tidak mengubah notebook/run sebelumnya. Sel aktif hanya membaca artefak dan memverifikasi rencana.
Tahap 1–3 belum diimplementasikan dan tidak dijalankan oleh Run All.

Pertanyaan: berapa biaya ekspansi vocabulary 6 → 8 pada kelas lama, dan apakah sepadan dengan halo/tidur?
Classification menjadi jalur utama sesuai keputusan peneliti. Detection tetap arsip/future work.
**Tidak ada janji B menang.** Null, non-inferior, inferior, dan inconclusive dilaporkan apa adanya.

Kernel: `thesis_cuthbert`, executable lokal `.venv-bisindo/Scripts/python.exe`.
Dokumen lengkap: [PREREGISTRATION.md](PREREGISTRATION.md), [DEVIATION_LOG.md](DEVIATION_LOG.md).
Status kandidat pra-registrasi: **menunggu persetujuan**. SHA256 kandidat bukan final confirmatory lock.

In [1]:
from pathlib import Path
import hashlib
import json
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
NOTEBOOK = ROOT / 'BISINDO_YOLO26_Vocabulary_Expansion_Research.ipynb'
PLAN = json.loads(NOTEBOOK.read_text(encoding='utf8'))['metadata']['bisindo_stage0']
assert PLAN['stage'] == 0 and PLAN['training_enabled'] is False
assert (ROOT / 'classification_pilot.py').is_file(), 'Open notebook from project root'
print('Python:', sys.version.split()[0])
print('Status:', PLAN['status'])
print('Training enabled:', PLAN['training_enabled'])

Python: 3.11.5
Status: AWAITING_STAGE0_APPROVAL
Training enabled: False


## 0A. Kondisi aktual — urutan otoritas artefak

Split/manifest/run records/predictions > analisis artefak > README > draft > usulan.
Sel berikut membaca metrik dan prediksi run lama, bukan menghasilkan hasil multi-seed/CV baru.
Notebook reproduksi lama adalah replikasi seed42; bukan seed independen kedua.

In [2]:
ORIGINAL = ROOT / 'experiment_yolo26_classification_pilot'
REPRODUCTION = ROOT / 'experiment_yolo26_classification_notebook_20261102'
metrics = pd.read_csv(ORIGINAL / 'evaluation/comparison_metrics.csv')
predictions = pd.read_csv(ORIGINAL / 'evaluation/predictions.csv')
manifest = pd.read_csv(ORIGINAL / 'manifest.csv', keep_default_na=False)
split = pd.read_csv(ROOT / 'experiment_yolo26_v3/split_lock.csv', keep_default_na=False)
assert not manifest.sample_id.duplicated().any()
assert manifest.source.eq('train').all()
assert manifest.groupby(['class_id','split']).size().unstack().eq(pd.Series({'train':60,'val':20})).all().all()
trained = json.loads((ORIGINAL / 'runs/trained_models.json').read_text(encoding='utf8'))
assert len(trained['models']) == 2 and all(r['epoch']==30 for r in trained['models'])
assert len({r['initial_checkpoint_sha256'] for r in trained['models']}) == 1
pd.testing.assert_frame_equal(metrics,pd.read_csv(REPRODUCTION / 'evaluation/comparison_metrics.csv'))
display(metrics[['model','scope','n','correct_n','accuracy','macro_f1','extra_class_predictions_n']])
display(split.groupby(['source','split']).size().rename('videos').reset_index())
display(manifest.groupby('quality_status').size().rename('frames').reset_index())
print('Known signer IDs:', int(manifest.signer.astype(str).str.strip().ne('').sum()))
print('Expert semantic decisions:', int(manifest.semantic_status.ne('not_assessed').sum()))
print('Recorded optimizer steps:', {r['model']:r['actual_optimizer_steps'] for r in trained['models']})

,model,scope,n,correct_n,accuracy,macro_f1,extra_class_predictions_n
0,A,common6,120,112,0.933333,0.931776,0
1,B,common6,120,111,0.925000,0.930042,2
2,B,full8,160,150,0.937500,0.935464,0


,source,split,videos
0,test,test,400
1,train,train,1920
2,train,val,480


,quality_status,frames
0,eligible,143
1,flagged,497


Known signer IDs: 0
Expert semantic decisions: 0
Recorded optimizer steps: {'A': 690, 'B': 900}


## 0B. Endpoint, hipotesis, pra-registrasi

- **H1 primer:** d = macro-F1_B(common6) − macro-F1_A(common6), non-inferiority margin **0.02**.
  NI memerlukan lower one-sided 95% CI > −0.02, bukan sekadar estimasi titik.
- **H2:** accuracy pada GT halo dan tidur masing-masing ≥ **80%**, disetujui peneliti.
  Usulan keputusan konfirmatori: lower CI per kelas >80%; target point estimate dilaporkan terpisah.
- **H3 sekunder:** D versus A_n pada faktorial yang juga menambah data lama; bukan efek kelas baru saja.
- Semua seed **42,43,44,45,46**. Sampling seed42 tetap. **last.pt** pada epoch tetap untuk primer.
- Equal epoch primer, equal steps suplemen; 690 steps hanya budget pilot A30/B23, bukan semua ukuran data.
- B tetap mengeluarkan delapan kelas pada common6. Full8 bukan tugas identik dengan common6.

Semantik frame/signer independence tidak diklaim. Val lama sudah dipakai diagnosis. CV seluruh internal development,
bukan independent confirmation. Domain konfirmatori yang diusulkan: satu batch external setelah freeze.
Kaggle pernah diproses di v2/v3; peneliti menyetujui sekali **evaluasi model final**, bukan klaim never-opened.

In [3]:
prereg_bytes = (ROOT / 'PREREGISTRATION.md').read_bytes()
prereg_hash = hashlib.sha256(prereg_bytes).hexdigest()
assert prereg_hash == PLAN['preregistration_sha256']
assert (ROOT / 'PREREGISTRATION.sha256').read_text(encoding='utf8').split()[0] == prereg_hash
print('Candidate protocol SHA256:', prereg_hash)
print('Candidate verified; activation/final freeze NOT created in Stage 0.')
display(pd.DataFrame(PLAN['grid']))
print('Nine proposed configurations; 90 CV A/B fits, not the full 27-configuration Cartesian grid.')

Candidate protocol SHA256: 24e21743168f3457933f0f17db76e52d420b5f005da774a7f9230c71f1ac7a70
Candidate verified; activation/final freeze NOT created in Stage 0.


,epochs,id,imgsz,regularization
0,10,G01,224,baseline
1,15,G02,224,baseline
2,30,G03,224,baseline
3,15,G04,320,baseline
4,30,G05,320,baseline
5,15,G06,448,baseline
6,30,G07,448,baseline
7,15,G08,320,label_smoothing_0.05
8,15,G09,320,proposed_label_safe


Nine proposed configurations; 90 CV A/B fits, not the full 27-configuration Cartesian grid.


## 0C. Rencana implementasi — Tahap 1

**Belum dijalankan.** Work baru `experiment_yolo26_classification_diag_v1/` setelah persetujuan.

1. Buat `analysis_paired.py` dan unit tests: exact McNemar, Wilson CI, paired video bootstrap,
   CSV/MD sembilan error dan per-class. Ini analisis exploratory, bukan confirmation.
2. Parameterisasi `classification_pilot.py` dan tes: seed/epochs/imgsz/work sampai preprocessing,
   release fingerprints, probe, transforms, evaluator, conclusion. Tidak ada flag baru yang sudah tersedia sekarang.
   Perintah lama tetap kompatibel dan menolak overwrite/resume.
3. Setelah izin training terpisah: 10 fits baru A/B ×5 seed, termasuk replikasi seed42;
   noise floor A42 vs A43..46 dari run yang sama, bukan pemilihan seed terbaik.
4. Kontrol hand-masked dengan landmark cache, missing landmarks dilaporkan tersendiri/tetap denominator.
   Ini context-only, bukan background murni. Chance balanced 1/6 atau 1/8; masking adalah distribution shift.
5. Frozen-backbone linear probes, 10 head fits, train lama saja, aturan sama. Tidak memakai validation untuk tuning head.
6. Kurva loss/top-1 per epoch. Train top-1 belum tercatat di run lama: perlu instrumentasi baru.

Stop keras jika technical gate gagal. Jika SD seed ≥ |mean d| atau CI melintasi margin,
pilot tidak cukup membedakan; usulkan CV dan minta izin, tanpa auto-launch.

## 0D. Rencana implementasi — Tahap 2

File fold baru atas 2.400 internal; split_lock asli tidak berubah.
Proxy grouping dari metadata container TEKNIS (resolusi, fps, codec, creation_time), tidak menebak signer dari penampilan.
StratifiedGroupKFold 5-fold bila feasible; fallback video-level StratifiedKFold memerlukan pengungkapan/persetujuan.
Group proxy bukan bukti independensi signer/sesi. Duplicate-content cluster tidak boleh menyeberang fold.

Faktorial: A_n old=n; B_n old=n/new=n; C old=2n; D old=2n/new=n.
Usulan n120 jika semua fold mampu 240/kelas; jika tidak, hash addendum kapasitas sebelum skor dilihat.
Subset nested, common6 validation identik semua arm. C/D memberi kontrol data lama penuh.

Grid 9 kandidat pada tabel, SGD tetap. Label smoothing perlu criterion implementasi/tes;
API terpasang tidak memiliki flag label_smoothing. G09 perlu validasi label-safe;
scale/translate tidak boleh memotong raw frame/tangan. Jangan menganggap parameter augmentasi otomatis diterapkan.

Selection satu config: mean fold dari rata-rata F1 common6 A_n/B_n, bukan selisih yang menguntungkan B.
Tie ≤1e-6: resolusi lebih kecil, epoch lebih sedikit, lalu grid ID terkecil.
Budget 90 tuning +10 C/D +5 B step-matched = **105 fits**.
OOF label-noise diagnosis tidak mengubah data utama. UI audit ahli tidak memperlihatkan prediksi/confidence/ranking.

## 0E. Rencana implementasi — Tahap 3

Setelah final freeze DAN izin: A_full old240, B_full old240/new240, lima seed = **10 fits primer**.
Jika faktorial final diaktifkan: A_n/B_n/D menambah **15 fits**; C memakai A_full yang sama.
D new120 berbeda dari B_full new240; jangan menyamakan keduanya. Total maksimal usulan final **25 fits**.

- Internal val reused dilaporkan supportive; Kaggle satu batch skor final (semua arm/seed terdaftar).
- External common6 300 adalah subset full8 400, bukan 700 video unik.
- Mean±SD d per seed, hierarchical paired video/seed bootstrap; video draw sama lintas model/seeds.
  Tidak menganggap seed×video sebagai sampel video independen.
- H1/H2 dahulu; H3 secondary gatekeeping. Tidak ada tuning setelah external score terbuka.
- Latency/FPS end-to-end preprocess+transfer+inference+postprocess diukur, bukan diasumsikan.
  Decode-inclusive pipeline dilaporkan terpisah; method, warmup, variance dan hardware tercatat.
- Audit noise berbasis random stratified semantic review yang diisi peneliti kemudian; targeted audit terpisah.

Draft baru nanti: **Effect of Vocabulary Expansion on YOLO26 Classification for BISINDO Phrase Recognition.md**.
Draft asli tidak ditimpa. Pisahkan PILOT EKSPLORATIF dari HASIL KONFIRMATORI yang belum ada.

In [4]:
budget = pd.DataFrame([
    {'phase':'0 planning/tests','full_fits':0,'head_fits':0,'estimated_wall_time':'No training'},
    {'phase':'1a analysis + implementation','full_fits':0,'head_fits':0,'estimated_wall_time':'15–45 min implementation/QA'},
    {'phase':'1b multi-seed + probes','full_fits':10,'head_fits':10,'estimated_wall_time':'15–60 min incl. preparation/QA'},
    {'phase':'2 CV grid','full_fits':90,'head_fits':0,'estimated_wall_time':'1–4 h + metadata preparation'},
    {'phase':'2 factorial + step supplement','full_fits':15,'head_fits':0,'estimated_wall_time':'20–90 min'},
    {'phase':'3 primary full data','full_fits':10,'head_fits':0,'estimated_wall_time':'10–60 min'},
    {'phase':'3 optional factorial extras','full_fits':15,'head_fits':0,'estimated_wall_time':'15–90 min'},
])
display(budget)
elapsed=[]
for model in ['A','B']:
    log=pd.read_csv(ORIGINAL / f'runs/model_{model}/results.csv')
    elapsed.append({'model':model,'epochs':len(log),'recorded_training_seconds':float(log.time.iloc[-1])})
display(pd.DataFrame(elapsed))
print('Estimates are proposals, not measured end-to-end benchmarks. Approval required for each training phase.')

,phase,full_fits,head_fits,estimated_wall_time
0,0 planning/tests,0,0,No training
1,1a analysis + implementation,0,0,15–45 min implementation/QA
2,1b multi-seed + probes,10,10,15–60 min incl. preparation/QA
3,2 CV grid,90,0,1–4 h + metadata preparation
4,2 factorial + step supplement,15,0,20–90 min
5,3 primary full data,10,0,10–60 min
6,3 optional factorial extras,15,0,15–90 min


,model,epochs,recorded_training_seconds
0,A,30,37.4169
1,B,30,47.5833


Estimates are proposals, not measured end-to-end benchmarks. Approval required for each training phase.


## 0F. Scope file dan persetujuan

**Dibuat Tahap 0:** notebook ini, PREREGISTRATION.md + SHA256 kandidat, DEVIATION_LOG.md,
serta test_stage0_research_plan.py. Source training, frozen split, human pilot, notebook dan hasil lama tetap.

**Setelah izin Tahap 1:** analysis_paired.py + test_analysis_paired.py; classification_pilot.py + test_classification_pilot.py;
sel notebook baru sesuai implementasi yang teruji. Kemudian dokumen paper baru. Fase CV/final dibuat terpisah
setelah hasil diagnosis, kapasitas fold, API dan budget diperiksa. Tidak ada asumsi bahwa fungsi itu sudah ada.

Sebelum run prospektif pertama, simpan persetujuan dan activation record/hash immutable.
Sebelum confirmation, freeze config winner/data/external manifests/source/checkpoint; jangan edit protokol lama.

In [5]:
changed=[]
for name, expected in PLAN['protected_artifact_sha256'].items():
    path=ROOT/name
    if not path.is_file() or hashlib.sha256(path.read_bytes()).hexdigest()!=expected:
        changed.append(name)
assert not changed, 'Protected artifacts changed: '+repr(changed)
print('PASS:',len(PLAN['protected_artifact_sha256']),'existing tracked artifact bytes preserved.')
print('STOP AT STAGE 0. Waiting for researcher approval before Stage 1 implementation/training.')

PASS: 182 existing tracked artifact bytes preserved.
STOP AT STAGE 0. Waiting for researcher approval before Stage 1 implementation/training.


## Batasan snapshot ini

Hasil lama hanya satu seed; tidak ada hasil baru McNemar/CI/bootstrap/multi-seed/CV/external atau latency di notebook ini.
Notebook rencana sengaja tidak memiliki pemanggilan training. Run All aman untuk inspeksi read-only Tahap 0.
Hasil pilot berasal dari label video lemah; signer independence, semantic verification dan real-time belum terbukti.